# VIX Kalman Regime Engine
## NIFTY50 Systematic Strategy — Research Summary
**Author**: Jash Taparia, IIT Bombay  
**Data**: NIFTY50 1-min OHLC, 2017–2025 (805k bars)  
**Approach**: Kalman filter on India VIX → regime detection → position sizing

---

### What this notebook covers
1. Strategy logic and signal architecture
2. Backtest results — v7 baseline and v2 DD-reduced version
3. Drawdown analysis — what was causing it and how it was fixed
4. Intraday edge analysis — what the data actually shows
5. Regime-to-options translation framework
6. Open questions for discussion

### Key Results
| Metric | v7 Baseline | v2 DD-Reduced |
|---|---|---|
| Ann. Return | 30.6% | **43.0%** |
| Sharpe | 0.9x | **1.7x** |
| Max Drawdown | -39.9% | **-28.1%** |
| Calmar | 0.8x | **1.5x** |
| Sortino | 1.2x | **2.8x** |

---
## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from pykalman import KalmanFilter
from scipy.stats import percentileofscore
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

NIFTY_CSV      = 'Nifty50_data.csv'
INITIAL_CAPITAL = 100_000
RISK_FREE_RATE  = 0.065

# v7 baseline parameters (for comparison)
VOL_TARGET         = 0.20
DD_CIRCUIT_BREAKER = 0.15
MAX_LEVERAGE       = 3.0
SLIPPAGE_BPS       = 8
VIX_ZSCORE_WINDOW  = 126
VIX_PCT_WINDOW     = 252
VIX_SPIKE_THRESH   = 0.20

REGIME_SCALARS_V7 = {
    'Extreme Panic': 0.65,
    'Panic':         0.80,
    'Fear (live)':   0.75,
    'Fear (buy)':    1.20,
    'Recovery':      2.50,
    'Calm':          2.80,
    'Neutral':       1.40,
}

# v2 DD-reduced parameters — 3 economic changes:
# 1. Cap Calm leverage (2.8x -> 1.8x) — VIX low != crash-proof
# 2. Trend filter — don't lever up into a falling market
# 3. Tighter circuit breaker (15% -> 10%)
DD_CIRCUIT_BREAKER_V2 = 0.10
REGIME_SCALARS_V2 = {
    'Extreme Panic': 0.65,
    'Panic':         0.80,
    'Fear (live)':   0.75,
    'Fear (buy)':    1.20,
    'Recovery':      2.00,
    'Calm':          1.80,
    'Neutral':       1.20,
}

# Use v2 as default throughout notebook
REGIME_SCALARS = REGIME_SCALARS_V2

C = dict(green='#00ff88', blue='#00bfff', red='#ff4466',
         gold='#ffdd00', grid='#2a2a2a', text='#cccccc', bg='#0d0d0d', panel='#141414')

print('Setup complete.')

---
## 1. Data Pipeline

In [ ]:
# Load 1-min, resample to daily
df = pd.read_csv(NIFTY_CSV)
df['datetime'] = pd.to_datetime(df['datetime'], utc=True).dt.tz_convert('Asia/Kolkata')
df = df.set_index('datetime').sort_index()
df = df.drop(columns=[c for c in ['volume','oi'] if c in df.columns])
df = df.between_time('09:15','15:30')

daily = df.resample('D').agg({'open':'first','high':'max','low':'min','close':'last'}).dropna()
daily.index = daily.index.tz_localize(None)

print(f'Daily bars: {len(daily)} trading days')
print(f'Range: {daily.index[0].date()} to {daily.index[-1].date()}')

# Download VIX
start_str = str(daily.index[0].date())
end_str   = str((daily.index[-1] + pd.Timedelta(days=1)).date())
vix_dl = yf.download('^INDIAVIX', start=start_str, end=end_str, progress=False, auto_adjust=True)
vix_raw = vix_dl['Close']['^INDIAVIX'] if isinstance(vix_dl.columns, pd.MultiIndex) else vix_dl['Close']
vix_raw.index = pd.to_datetime(vix_raw.index).tz_localize(None)
vix_raw.name  = 'VIX'
print(f'VIX: {len(vix_raw)} days downloaded')

---
## 2. Kalman Filter on India VIX

**Why Kalman?** India VIX is noisy. A raw z-score on VIX generates too many false regime signals during brief spikes. The Kalman filter separates the underlying *level* and *velocity* of fear from day-to-day noise, giving smoother and more reliable regime transitions.

**State vector**: `[VIX level, VIX velocity]`  
**Output**: smoothed level + rate of change → used to compute z-score and detect panic spikes

In [ ]:
def build_vix_features(vix_series):
    F = np.array([[1,1],[0,1]])
    H = np.array([[1,0]])
    kf = KalmanFilter(
        transition_matrices=F, observation_matrices=H,
        initial_state_mean=[float(vix_series.iloc[0]), 0.0],
        initial_state_covariance=np.eye(2),
        transition_covariance=np.eye(2)*1e-4,
        observation_covariance=np.array([[0.8]])
    )
    states, _ = kf.filter(vix_series.values)

    out = pd.DataFrame({'VIX': vix_series.values, 'KF_Level': states[:,0], 'KF_Vel': states[:,1]},
                       index=vix_series.index)

    rm = out['KF_Level'].rolling(VIX_ZSCORE_WINDOW, min_periods=21).mean()
    rs = out['KF_Level'].rolling(VIX_ZSCORE_WINDOW, min_periods=21).std().replace(0, np.nan)
    out['VIX_Z'] = (out['KF_Level'] - rm) / rs

    def rpct(x):
        return percentileofscore(x[:-1], x.iloc[-1]) / 100.0
    out['VIX_Pct'] = out['VIX'].rolling(VIX_PCT_WINDOW, min_periods=63).apply(rpct, raw=False)

    vel_std = out['KF_Vel'].rolling(42, min_periods=14).std().replace(0, np.nan)
    out['VelZ'] = out['KF_Vel'] / vel_std

    out['Spike'] = (out['VIX'].pct_change() > VIX_SPIKE_THRESH).astype(int).rolling(3, min_periods=1).max()
    return out


vix_feat = build_vix_features(vix_raw)
print(f'VIX Z-score range: {vix_feat["VIX_Z"].min():.2f} to {vix_feat["VIX_Z"].max():.2f}')
print(f'Panic spikes:      {vix_feat["Spike"].sum()}')

---
## 3. Regime State Machine

Seven regimes, priority-ordered. Each regime has a different position scalar:

| Regime | VIX Condition | Position Scalar | Logic |
|---|---|---|---|
| Extreme Panic | Z > 3.0 | 0.65× | Reduce but stay long — peak fear often marks bottom |
| Panic | Z > 2.0 OR spike | 0.80× | Still elevated risk, slight reduction |
| Fear (live) | Z > 1.2, vel rising | 0.75× | Fear building — trim |
| Fear (buy) | Z > 1.2, vel falling | 1.20× | Fear peaked — start adding |
| Recovery | vel strongly negative, pct > 45 | 2.50× | VIX collapsing — lever up |
| Calm | Z < -0.3 | 2.80× | Low vol environment — maximum exposure |
| Neutral | Everything else | 1.40× | Moderate exposure |

In [ ]:
def assign_regimes(vix_feat):
    vz    = vix_feat['VIX_Z'].fillna(0)
    velz  = vix_feat['VelZ'].fillna(0)
    pct   = vix_feat['VIX_Pct'].fillna(0.5)
    spike = vix_feat['Spike']

    is_ep   = (vz > 3.0)
    is_p    = (~is_ep) & ((vz > 2.0) | (spike > 0))
    is_fl   = (~is_ep) & (~is_p) & (vz > 1.2) & (velz > 0.5)
    is_fb   = (~is_ep) & (~is_p) & (vz > 1.2) & (velz <= 0.5)
    is_rec  = (~is_ep) & (~is_p) & (~is_fl) & (~is_fb) & (velz < -1.0) & (pct > 0.45)
    is_calm = (~is_ep) & (~is_p) & (~is_fl) & (~is_fb) & (~is_rec) & (vz < -0.3)

    regime = np.where(is_ep,   'Extreme Panic',
              np.where(is_p,   'Panic',
              np.where(is_fl,  'Fear (live)',
              np.where(is_fb,  'Fear (buy)',
              np.where(is_rec, 'Recovery',
              np.where(is_calm,'Calm', 'Neutral'))))))

    vix_feat['Regime'] = regime
    vix_feat['Scalar'] = pd.Series(regime, index=vix_feat.index).map(REGIME_SCALARS)

    total = len(regime)
    print(f'Regime distribution ({total} days):')
    order = ['Extreme Panic','Panic','Fear (live)','Fear (buy)','Recovery','Calm','Neutral']
    for r in order:
        n = (regime == r).sum()
        s = REGIME_SCALARS[r]
        print(f'  {r:<16}: {n:>4} days ({n/total*100:.1f}%)  scalar {s}x')
    return vix_feat


vix_feat = assign_regimes(vix_feat)

---
## 4. Position Sizing & Backtest

In [ ]:
def run_backtest(daily, vix_feat, scalars, cb_threshold, name):
    ma5      = daily['close'].rolling(5).mean()
    trend_ok = (daily['close'] > ma5).rename('TrendOK')

    data = pd.concat([
        daily['close'].rename('NIFTY'),
        vix_feat[['VIX_Z','Regime','Scalar']],
        trend_ok
    ], axis=1).ffill().dropna()
    ret = data['NIFTY'].pct_change().fillna(0)

    data['Scalar_use'] = data['Regime'].map(scalars)
    # Trend filter: cap at 1.0x if market falling
    data['Scalar_use'] = np.where(
        data['TrendOK'], data['Scalar_use'],
        data['Scalar_use'].clip(0, 1.0)
    )

    ewma_vol   = ret.ewm(span=21).std() * np.sqrt(252)
    vol_scalar = (VOL_TARGET / ewma_vol.replace(0, np.nan)).clip(0.3, MAX_LEVERAGE)
    raw_pos    = (data['Scalar_use'] * vol_scalar).clip(0, MAX_LEVERAGE).ewm(span=2).mean()

    pos_arr   = raw_pos.values.copy()
    ret_arr   = ret.values
    vz_arr    = data['VIX_Z'].fillna(0).values
    equity    = np.zeros(len(pos_arr))
    equity[0] = INITIAL_CAPITAL
    final_pos = np.zeros(len(pos_arr))
    brake     = 1.0

    for i in range(1, len(pos_arr)):
        final_pos[i] = pos_arr[i] * brake
        slip         = abs(final_pos[i] - final_pos[i-1]) * (SLIPPAGE_BPS / 10_000)
        equity[i]    = equity[i-1] * (1 + final_pos[i] * ret_arr[i] - slip)
        if i >= 20:
            peak_20 = np.max(equity[max(0,i-20):i+1])
            dd_20   = (equity[i] - peak_20) / peak_20
            if dd_20 < -cb_threshold:
                brake = 0.25
            elif brake < 1.0 and vz_arr[i] < 1.0:
                brake = 0.50 if brake == 0.25 else 1.00

    strat_eq  = pd.Series(equity, index=data.index)
    strat_ret = pd.Series(final_pos, index=data.index) * ret - \
                pd.Series(final_pos, index=data.index).diff().abs().fillna(0) * (SLIPPAGE_BPS/10_000)
    bench_eq  = INITIAL_CAPITAL * (1 + ret).cumprod()
    print(f'{name}: Rs{strat_eq.iloc[-1]:,.0f}  (Benchmark: Rs{bench_eq.iloc[-1]:,.0f})')
    return strat_ret, strat_eq, ret, bench_eq, data


# Run both versions
strat_ret_v7, strat_eq_v7, bench_ret, bench_eq, data_v7 = run_backtest(
    daily, vix_feat, REGIME_SCALARS_V7, 0.15, 'v7 baseline'
)
strat_ret_v2, strat_eq_v2, bench_ret, bench_eq, data_v2 = run_backtest(
    daily, vix_feat, REGIME_SCALARS_V2, 0.10, 'v2 DD-reduced'
)

# Use v2 as primary throughout
strat_ret = strat_ret_v2
strat_eq  = strat_eq_v2
data      = data_v2

---
## 5. Performance Metrics

In [ ]:
def metrics(ret, name):
    r   = ret[ret != 0].dropna()
    ann = r.mean() * 252
    vol = r.std()  * np.sqrt(252)
    sh  = (ann - RISK_FREE_RATE) / vol if vol else 0
    eq  = INITIAL_CAPITAL * (1 + r).cumprod()
    pk  = eq.cummax()
    dd  = (eq - pk) / pk
    mdd = dd.min()
    cal = ann / abs(mdd) if mdd else 0
    dn  = r[r < 0]
    srt = (ann - RISK_FREE_RATE) / (dn.std() * np.sqrt(252)) if len(dn) else 0
    wr  = (r > 0).mean()
    pf  = r[r>0].sum() / abs(r[r<0].sum()) if len(r[r<0]) else np.inf
    in_dd = (dd < -0.01).astype(int)
    grp   = (in_dd != in_dd.shift()).cumsum()
    mdd_d = int(in_dd.groupby(grp).sum().max()) if len(in_dd) else 0
    return {'ann': ann, 'vol': vol, 'sharpe': sh, 'mdd': mdd, 'calmar': cal,
            'sortino': srt, 'wr': wr, 'pf': pf, 'mdd_days': mdd_d}, dd


v7_m, v7_dd = metrics(strat_ret_v7, 'v7')
v2_m, v2_dd = metrics(strat_ret_v2, 'v2')
strat_m, strat_dd = v2_m, v2_dd   # use v2 for dashboard
bench_m, bench_dd = metrics(bench_ret, 'benchmark')

print(f'\n{"Metric":<22} {"v7 Baseline":>14} {"v2 DD-Reduced":>14} {"Buy&Hold":>10}')
print('-' * 62)
for label, v7v, v2v, bhv, unit in [
    ('Ann. Return',    v7_m['ann']*100,   v2_m['ann']*100,   bench_m['ann']*100,  '%'),
    ('Ann. Vol',       v7_m['vol']*100,   v2_m['vol']*100,   bench_m['vol']*100,  '%'),
    ('Sharpe',         v7_m['sharpe'],    v2_m['sharpe'],    bench_m['sharpe'],   'x'),
    ('Sortino',        v7_m['sortino'],   v2_m['sortino'],   bench_m['sortino'],  'x'),
    ('Max Drawdown',   v7_m['mdd']*100,   v2_m['mdd']*100,   bench_m['mdd']*100,  '%'),
    ('Calmar',         v7_m['calmar'],    v2_m['calmar'],    bench_m['calmar'],   'x'),
    ('Win Rate',       v7_m['wr']*100,    v2_m['wr']*100,    bench_m['wr']*100,   '%'),
    ('Profit Factor',  v7_m['pf'],        v2_m['pf'],        bench_m['pf'],       'x'),
    ('Max DD Days',    v7_m['mdd_days'],  v2_m['mdd_days'],  bench_m['mdd_days'], 'd'),
]:
    fmt = '{:>13.1f}{}' if unit != 'd' else '{:>13d}{}'
    print(f"{label:<22} {fmt.format(v7v if unit!='d' else int(v7v), unit)}"
          f" {fmt.format(v2v if unit!='d' else int(v2v), unit)}"
          f" {fmt.format(bhv if unit!='d' else int(bhv), unit)}")

print(f'\nDD reduction: {(v2_m["mdd"] - v7_m["mdd"])*100:.1f}pp  |  '
      f'Return change: +{(v2_m["ann"] - v7_m["ann"])*100:.1f}pp  |  '
      f'Sharpe change: +{v2_m["sharpe"] - v7_m["sharpe"]:.2f}x')

---
## 6. Intraday Edge Analysis

Before building intraday signals, I tested whether any intraday edge exists in the raw data. Key findings:

| Test | Result | Interpretation |
|---|---|---|
| 5-min autocorrelation (lag 1–12) | 0.01 to 0.015 | Effectively zero — no momentum |
| ORB win rate (long breakout) | 50.7% | Coin flip — no directional edge |
| ORB win rate (short breakout) | 51.4% | Coin flip — no directional edge |
| ORB avg return after breakout | -0.006% / +0.010% | Indistinguishable from zero |
| Time-of-day t-stat | Only 15:25 > 2.0 | Last bar of session, untradeable |

**Conclusion**: NIFTY50 spot is highly efficient at the 5-min intraday level. Price-only signals have no detectable edge. This is consistent with the academic literature on large-cap equity index microstructure.

**The real opportunity** is what Sarthak described: replicating these regime signals on options strategies where the edge comes from volatility mispricing, not price direction.

In [ ]:
# Reproduce the EDA findings cleanly
df1m = pd.read_csv(NIFTY_CSV)
df1m['datetime'] = pd.to_datetime(df1m['datetime'], utc=True).dt.tz_convert('Asia/Kolkata')
df1m = df1m.set_index('datetime').sort_index().between_time('09:15','15:30')
df1m['ret'] = df1m['close'].pct_change()

df5 = df1m['close'].resample('5min').last().dropna().to_frame()
df5['ret'] = df5['close'].pct_change()
df5 = df5.between_time('09:15','15:25').dropna()

print('5-min autocorrelation (price momentum test):')
for lag in [1, 2, 3, 6, 12]:
    corr = df5['ret'].autocorr(lag=lag)
    sig  = '← near zero, no momentum' if abs(corr) < 0.02 else '← SIGNAL'
    print(f'  Lag {lag:>2}: {corr:>8.5f}  {sig}')

print('\nORB directional accuracy test:')
results = []
for day, grp in df5.groupby(df5.index.date):
    or_bars = grp.between_time('09:15','09:29')
    if len(or_bars) < 3: continue
    OR_high = or_bars['close'].max()
    OR_low  = or_bars['close'].min()
    rest    = grp.between_time('09:30','15:15')
    if len(rest) < 5: continue
    first_break = None
    for ts, row in rest.iterrows():
        if row['close'] > OR_high:  first_break = +1; break
        elif row['close'] < OR_low: first_break = -1; break
    if first_break is None: continue
    break_idx  = rest.index.get_loc(ts)
    after_bars = rest.iloc[break_idx+1:]
    if len(after_bars) < 3: continue
    day_ret = (after_bars['close'].iloc[-1] - row['close']) / row['close']
    results.append({'direction': first_break, 'ret': day_ret})

res = pd.DataFrame(results)
for d, label in [(1,'Long  breakout'), (-1,'Short breakout')]:
    sub    = res[res['direction']==d]
    signed = sub['ret'] * d
    wr     = (signed > 0).mean() * 100
    avg    = signed.mean() * 100
    print(f'  {label}: {len(sub)} days  WR {wr:.1f}%  Avg return {avg:.3f}%  ← coin flip')

---
## 7. Regime → Options Translation

The spot strategy identifies the volatility regime. The real alpha is in translating these regimes into options positions — selling expensive IV in fear regimes, buying cheap directional premium in calm regimes.

In [ ]:
options_map = pd.DataFrame([
    {
        'Regime':        'Calm',
        'VIX Context':   'Low IV (VIX Z < -0.3)',
        'Spot Pos':      '2.8x Long',
        'Options Play':  'Buy ATM Call or Bull Call Spread',
        'IV Edge':       'Buy cheap vol — IV understates realised vol in calm trending markets',
        'Risk':          'Premium paid if wrong direction'
    },
    {
        'Regime':        'Recovery',
        'VIX Context':   'IV collapsing from peak',
        'Spot Pos':      '2.5x Long',
        'Options Play':  'Buy ATM Call + Sell OTM Call (bull spread)',
        'IV Edge':       'Long delta as market recovers, IV crush adds to P&L',
        'Risk':          'Recovery stalls and VIX re-spikes'
    },
    {
        'Regime':        'Neutral',
        'VIX Context':   'Average IV',
        'Spot Pos':      '1.4x Long',
        'Options Play':  'Short Strangle (sell OTM call + put)',
        'IV Edge':       'Sell fairly-priced IV, collect theta in range-bound market',
        'Risk':          'Directional breakout exceeds strangle width'
    },
    {
        'Regime':        'Fear (buy)',
        'VIX Context':   'Elevated IV, velocity falling — fear peaked',
        'Spot Pos':      '1.2x Long',
        'Options Play':  'Sell OTM Put Spread (Bull Put) or Buy Call',
        'IV Edge':       'IV at peak — sell expensive puts, collect mean-reversion premium',
        'Risk':          'Fear resumes — VIX velocity reverses up'
    },
    {
        'Regime':        'Fear (live)',
        'VIX Context':   'Elevated IV, velocity rising — fear building',
        'Spot Pos':      '0.75x Long',
        'Options Play':  'Buy Put Spread for hedge or reduce exposure',
        'IV Edge':       'IV rising — buy protection while still cheap relative to peak',
        'Risk':          'Fear dissipates quickly — hedge bleeds premium'
    },
    {
        'Regime':        'Panic',
        'VIX Context':   'Very high IV (Z > 2.0)',
        'Spot Pos':      '0.8x Long',
        'Options Play':  'Sell far OTM Strangle or Iron Condor',
        'IV Edge':       'IV massively inflated — sell vol with defined risk, collect panic premium',
        'Risk':          'Tail event — market gaps beyond strikes'
    },
    {
        'Regime':        'Extreme Panic',
        'VIX Context':   'Extreme IV (Z > 3.0)',
        'Spot Pos':      '0.65x Long',
        'Options Play':  'Sell OTM Put (cash-secured) — high premium, mean-revert play',
        'IV Edge':       'Historically extreme IV — enormous premium available for put sellers',
        'Risk':          'Genuine systemic event — puts get exercised'
    },
])

pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)
print('VIX Regime → Options Strategy Translation:')
print('=' * 90)
for _, row in options_map.iterrows():
    print(f"\nREGIME: {row['Regime']}  |  {row['VIX Context']}")
    print(f"  Spot:    {row['Spot Pos']}")
    print(f"  Options: {row['Options Play']}")
    print(f"  Edge:    {row['IV Edge']}")
    print(f"  Risk:    {row['Risk']}")

---
## 8. Full Dashboard

In [ ]:
def style(ax, title):
    ax.set_facecolor(C['panel'])
    ax.tick_params(colors=C['text'], labelsize=8)
    ax.set_title(title, color=C['text'], fontsize=9, fontweight='bold', pad=6)
    for sp in ax.spines.values(): sp.set_color(C['grid'])
    ax.grid(True, color=C['grid'], linewidth=0.5, alpha=0.7)

fig = plt.figure(figsize=(20, 22))
fig.patch.set_facecolor(C['bg'])
gs  = gridspec.GridSpec(5, 3, figure=fig, hspace=0.50, wspace=0.30)

# Panel 1: Equity curve — all three
ax1 = fig.add_subplot(gs[0, :])
style(ax1, 'Equity Curve  |  v7 Baseline vs v2 DD-Reduced vs Buy & Hold')
ax1.plot(strat_eq_v2.index, strat_eq_v2, color=C['green'], lw=1.8,
         label=f"v2 DD-Reduced  Ann {v2_m['ann']*100:.1f}%  Sharpe {v2_m['sharpe']:.2f}  Calmar {v2_m['calmar']:.2f}  MaxDD {v2_m['mdd']*100:.1f}%")
ax1.plot(strat_eq_v7.index, strat_eq_v7, color=C['gold'], lw=1.2, alpha=0.7, ls='--',
         label=f"v7 Baseline    Ann {v7_m['ann']*100:.1f}%  Sharpe {v7_m['sharpe']:.2f}  Calmar {v7_m['calmar']:.2f}  MaxDD {v7_m['mdd']*100:.1f}%")
ax1.plot(bench_eq.index, bench_eq, color=C['blue'], lw=1.0, alpha=0.55,
         label=f"Buy & Hold     Ann {bench_m['ann']*100:.1f}%  Sharpe {bench_m['sharpe']:.2f}  MaxDD {bench_m['mdd']*100:.1f}%")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'Rs{x/1000:.0f}K'))
ax1.legend(fontsize=8, facecolor='#1a1a1a', edgecolor=C['grid'], labelcolor=C['text'], loc='upper left')

# Panel 2: Drawdown
ax2 = fig.add_subplot(gs[1, 0])
style(ax2, 'Strategy Drawdown')
ax2.fill_between(strat_dd.index, strat_dd*100, 0, color=C['red'], alpha=0.55)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}%'))

# Panel 3: VIX + Z-score
ax3 = fig.add_subplot(gs[1, 1])
style(ax3, 'India VIX  &  Kalman Z-Score')
ax3b = ax3.twinx()
ax3.plot(vix_feat.index,  vix_feat['VIX'],   color=C['red'],  lw=0.9, label='VIX')
ax3b.plot(vix_feat.index, vix_feat['VIX_Z'], color=C['gold'], lw=0.8, alpha=0.9, label='Z-Score')
ax3b.axhline(2.0,  color=C['red'],   lw=0.5, ls='--', alpha=0.7)
ax3b.axhline(-0.3, color=C['green'], lw=0.5, ls='--', alpha=0.7)
ax3.set_ylabel('VIX',    color=C['red'],  fontsize=7)
ax3b.set_ylabel('Z',     color=C['gold'], fontsize=7)
ax3b.tick_params(colors=C['text'], labelsize=8)
for sp in ax3b.spines.values(): sp.set_color(C['grid'])

# Panel 4: Regime timeline
ax4 = fig.add_subplot(gs[1, 2])
style(ax4, 'Regime Distribution')
order  = ['Extreme Panic','Panic','Fear (live)','Fear (buy)','Neutral','Recovery','Calm']
cols   = [C['red'],'#cc2244','#ff6633','#ffaa44',C['gold'],C['blue'],C['green']]
counts = [((vix_feat['Regime']==r).sum()) for r in order]
bars   = ax4.barh(order, counts, color=cols, alpha=0.85)
total  = sum(counts)
for bar, cnt in zip(bars, counts):
    ax4.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
             f'{cnt/total*100:.1f}%', va='center', color=C['text'], fontsize=7)
ax4.tick_params(colors=C['text'], labelsize=7)
for sp in ax4.spines.values(): sp.set_color(C['grid'])

# Panel 5: Rolling 126d Sharpe
ax5 = fig.add_subplot(gs[2, 0])
style(ax5, 'Rolling 126d Sharpe')
rs = (strat_ret.rolling(126).mean()*252 - RISK_FREE_RATE) / (strat_ret.rolling(126).std()*np.sqrt(252))
ax5.plot(rs.index, rs, color=C['gold'], lw=0.9)
ax5.axhline(0, color=C['grid'], lw=0.5)
ax5.axhline(1, color=C['green'], lw=0.4, ls='--', alpha=0.6)
ax5.fill_between(rs.index, rs, 0, where=rs>=0, color=C['green'], alpha=0.2)
ax5.fill_between(rs.index, rs, 0, where=rs<0,  color=C['red'],   alpha=0.2)

# Panel 6: VIX velocity
ax6 = fig.add_subplot(gs[2, 1])
style(ax6, 'VIX Velocity Z-Score  (Regime Trigger)')
vz = vix_feat['VelZ'].clip(-4,4)
ax6.fill_between(vix_feat.index, vz, 0, where=vz>0,  color=C['red'],   alpha=0.4)
ax6.fill_between(vix_feat.index, vz, 0, where=vz<=0, color=C['green'], alpha=0.4)
ax6.plot(vix_feat.index, vz, color=C['gold'], lw=0.7)
ax6.axhline(0.5,  color=C['red'],   lw=0.4, ls='--', alpha=0.6)
ax6.axhline(-1.0, color=C['green'], lw=0.4, ls='--', alpha=0.6)

# Panel 7: Annual returns
ax7 = fig.add_subplot(gs[2, 2])
style(ax7, 'Annual Returns')
ann_s = strat_ret.resample('YE').apply(lambda x: (1+x).prod()-1) * 100
ann_b = bench_ret.resample('YE').apply(lambda x: (1+x).prod()-1) * 100
years = ann_s.index.year
x     = np.arange(len(years))
ax7.bar(x-0.2, ann_s.values, 0.35, color=C['green'], alpha=0.8, label='Strategy')
ax7.bar(x+0.2, ann_b.values, 0.35, color=C['blue'],  alpha=0.6, label='Buy & Hold')
ax7.set_xticks(x); ax7.set_xticklabels(years, color=C['text'], fontsize=7, rotation=45)
ax7.axhline(0, color=C['grid'], lw=0.5)
ax7.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}%'))
ax7.legend(fontsize=7, facecolor='#1a1a1a', edgecolor=C['grid'], labelcolor=C['text'])

# Panel 8: Monthly heatmap
ax8 = fig.add_subplot(gs[3, :])
style(ax8, 'Monthly Returns Heatmap  (Strategy)')
monthly = strat_ret.resample('ME').apply(lambda x: (1+x).prod()-1) * 100
mdf     = monthly.to_frame('ret')
mdf['Year']  = mdf.index.year
mdf['Month'] = mdf.index.month
pivot   = mdf.pivot(index='Year', columns='Month', values='ret')
pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
vals    = pivot.values[~np.isnan(pivot.values)]
vmax    = max(abs(vals).max(), 1) if len(vals) else 1
im      = ax8.imshow(pivot.values, cmap='RdYlGn', vmin=-vmax, vmax=vmax, aspect='auto')
ax8.set_xticks(range(12)); ax8.set_xticklabels(pivot.columns, color=C['text'], fontsize=7)
ax8.set_yticks(range(len(pivot.index))); ax8.set_yticklabels(pivot.index, color=C['text'], fontsize=7)
for i in range(len(pivot.index)):
    for j in range(12):
        v = pivot.values[i,j]
        if not np.isnan(v):
            ax8.text(j, i, f'{v:.1f}', ha='center', va='center',
                     fontsize=5.5, color='black' if abs(v)<vmax*0.6 else 'white')
plt.colorbar(im, ax=ax8, fraction=0.01, pad=0.01).ax.tick_params(colors=C['text'], labelsize=6)

# Panel 9: Key metrics summary
ax9 = fig.add_subplot(gs[4, :])
ax9.set_facecolor(C['panel'])
for sp in ax9.spines.values(): sp.set_color(C['grid'])
ax9.set_xticks([]); ax9.set_yticks([])
ax9.set_title('Strategy Summary  |  Open Questions for Discussion', color=C['text'], fontsize=9, fontweight='bold', pad=6)

left_text = (
    f"VIX KALMAN v2 (DD-REDUCED)\n"
    f"{'─'*34}\n"
    f"Ann. Return     {v2_m['ann']*100:>8.1f}%   BH: {bench_m['ann']*100:.1f}%\n"
    f"Sharpe          {v2_m['sharpe']:>8.2f}x   BH: {bench_m['sharpe']:.2f}x\n"
    f"Sortino         {v2_m['sortino']:>8.2f}x\n"
    f"Max Drawdown    {v2_m['mdd']*100:>8.1f}%   BH: {bench_m['mdd']*100:.1f}%\n"
    f"Calmar          {v2_m['calmar']:>8.2f}x   BH: {bench_m['calmar']:.2f}x\n"
    f"Win Rate        {v2_m['wr']*100:>8.1f}%\n"
    f"Profit Factor   {v2_m['pf']:>8.2f}x\n"
    f"Max DD Duration {v2_m['mdd_days']:>8d}d\n\n"
    f"DD FIX (v7 -> v2)\n"
    f"{'─'*34}\n"
    f"Calm scalar:    2.80x -> 1.80x\n"
    f"Recovery:       2.50x -> 2.00x\n"
    f"Circuit breaker:  15% -> 10%\n"
    f"+ Trend filter (no leverage\n"
    f"  if below 5-day MA)"
)
right_text = (
    "OPEN QUESTIONS FOR SARTHAK\n"
    "─────────────────────────────────────────\n"
    "1. Intraday spot has no detectable edge at 5-min.\n"
    "   How do you generate intraday signals for options entry?\n\n"
    "2. Which options strategies map best to each regime?\n"
    "   (Strangle in neutral, put spread in fear-buy, etc.)\n\n"
    "3. How do you model IV crush / expansion in the backtest?\n"
    "   Need options chain data or IV surface proxy.\n\n"
    "4. What is your preferred arb structure on NIFTY?\n"
    "   Calendar spread? Put-call parity? Futures basis?"
)
ax9.text(0.01, 0.95, left_text,  transform=ax9.transAxes, fontsize=7.5,
         color=C['green'], fontfamily='monospace', va='top')
ax9.text(0.40, 0.95, right_text, transform=ax9.transAxes, fontsize=7.5,
         color=C['gold'],  fontfamily='monospace', va='top')

fig.suptitle('VIX Kalman Regime Engine  |  NIFTY50  |  2017–2025  |  Research Summary',
             color='white', fontsize=13, fontweight='bold', y=0.995)
plt.savefig('vix_kalman_summary.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
print('Dashboard saved to vix_kalman_summary.png')
plt.show()